# Deep Learning with Keras — Classification & Regression

**Companion notebook for the *Deep Learning & Artificial Neural Networks* session.**

We build two small feedforward networks from scratch:

| Task | Dataset | Question |
|---|---|---|
| **Classification** | Breast cancer (Wisconsin) | Is this tumour malignant or benign? |
| **Regression** | California housing | What is the median house value of this district? |

Everything here is the theory from the session, in code: the architecture choices from
**Part 03**, the output unit and cost function from **Part 04**, dropout and early
stopping from **Part 05**, and the Adam optimizer from **Part 06**.

**Estimated time:** 40–50 min · **Runtime:** CPU is fine, no GPU needed

## 0 · Setup

In [ ]:
# Every import in this notebook lives in this one cell.
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             mean_absolute_error, mean_squared_error, r2_score)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("You're all set!")

## 1 · The workflow

Every model in this notebook follows the same five steps. This is the whole session in
one list — the rest is detail.

1. **Get the data**, then split it into *train / validation / test*.
   The validation set tunes our choices; the test set is touched exactly once, at the end.
2. **Standardize the features.** Gradient descent struggles on badly scaled inputs
   (this is the *ill-conditioning* problem from Part 06).
3. **Build the network** — a chain of `Dense` layers with ReLU activations,
   and an output layer whose shape is dictated by the task.
4. **Compile**: choose the loss (from the output distribution) and the optimizer.
5. **Fit** with early stopping, then **evaluate** on the test set.

> **The one rule that matters most:** fit the scaler on the *training* data only.
> If you fit it on everything, information from the test set leaks into training and
> your reported score becomes optimistic — a subtle bug that is easy to miss.

### A helper for learning curves

We will plot training vs. validation curves for both models, so let's write that once.
These plots are exactly the figure from Part 05 — the gap between the two curves *is*
the generalization gap, and the point where the validation curve turns is where early
stopping should fire.

In [ ]:
def plot_history(history, metric, title):
    """Plot train vs validation curves for the loss and one other metric."""
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

    for ax, key in zip(axes, ["loss", metric]):
        ax.plot(h[key], label=f"train {key}")
        ax.plot(h["val_" + key], label=f"validation {key}")
        best = int(np.argmin(h["val_loss"]))
        ax.axvline(best, color="grey", ls="--", lw=1)
        ax.set_xlabel("epoch")
        ax.set_ylabel(key)
        ax.legend()
        ax.grid(alpha=.3)

    axes[0].set_title(f"{title} — loss")
    axes[1].set_title(f"{title} — {metric}")
    plt.tight_layout()
    plt.show()
    print(f"Best epoch (lowest validation loss): {int(np.argmin(h['val_loss'])) + 1}")

## 2 · Classification — breast cancer

569 samples, 30 numeric features computed from a digitised image of a breast mass
(radius, texture, concavity, and so on). The target is binary:
**0 = malignant, 1 = benign**.

Because the target is a single binary variable, the output distribution is **Bernoulli** —
which, from Part 04, fixes the rest of our choices: one output unit, a **sigmoid**
activation, and **binary cross-entropy** as the loss.

### 2.1 Load and inspect

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

print("Feature matrix:", X.shape)
print("Targets:       ", y.shape)
print("Classes:       ", dict(zip(cancer.target_names, np.bincount(y))))
print("\nFirst 5 feature names:", list(cancer.feature_names[:5]))
print("\nFeature scales differ wildly — look at the ranges:")
for i in [0, 3, 23]:
    print(f"  {cancer.feature_names[i]:<24} {X[:, i].min():8.2f} to {X[:, i].max():8.2f}")

Notice that last block: one feature ranges over single digits while another spans
thousands. Feeding that straight into a network means the loss surface is stretched into
a long narrow valley — precisely the ill-conditioned case where a single learning rate
cannot serve every direction. Standardizing fixes it.

### 2.2 Split, then scale

In [ ]:
# 60% train / 20% validation / 20% test, preserving the class balance
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=SEED, stratify=y_tmp)

# Fit the scaler on TRAINING data only, then apply it to all three splits
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"train {X_train.shape[0]}   val {X_val.shape[0]}   test {X_test.shape[0]}")
print(f"train mean ~ {X_train.mean():.3f}, train std ~ {X_train.std():.3f}")

### 2.3 Build the network

A chain of three functions, exactly as in Part 03: two hidden layers and an output layer.

- **ReLU** on the hidden units — the default recommendation from Part 04.
- **Dropout(0.3)** after each hidden layer. During training each unit is kept with
  probability 0.7; at prediction time Keras turns dropout off and rescales automatically
  (the *inverted dropout* trick from Part 05 — you never have to do it by hand).
- **One sigmoid output**, because we are modelling a Bernoulli distribution.

In [ ]:
def build_classifier(n_features, dropout=0.3):
    model = keras.Sequential([
        keras.Input(shape=(n_features,)),
        layers.Dense(32, activation="relu", kernel_initializer="he_normal"),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_initializer="he_normal"),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ], name="breast_cancer_classifier")
    return model

clf = build_classifier(X_train.shape[1])
clf.summary()

`he_normal` is the He initialization from Part 06 — a zero-centred Gaussian with variance
`2/m`, the scheme designed for ReLU units. Keras defaults to Glorot (Xavier), which is the
right choice for tanh or sigmoid; with ReLU hidden units, He is the better match.

### 2.4 Compile

In [ ]:
clf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")],
)
print("Loss:      binary cross-entropy  (Bernoulli output -> negative log-likelihood)")
print("Optimizer: Adam, lr = 0.001      (the suggested default from Part 06)")

### 2.5 Train with early stopping

`EarlyStopping` is the algorithm from Part 05, implemented for us: it watches the
validation loss, remembers the parameters that produced the best value, and stops once
`patience` epochs pass with no improvement. `restore_best_weights=True` is the important
flag — without it you keep the *last* weights rather than the *best* ones.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,             # how long to wait for an improvement
    restore_best_weights=True,
    verbose=1,
)

history_clf = clf.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=300,              # an upper bound; early stopping decides the real number
    batch_size=32,           # mini-batch SGD, Part 06
    callbacks=[early_stop],
    verbose=0,
)
print(f"Stopped after {len(history_clf.history['loss'])} epochs.")

In [ ]:
plot_history(history_clf, metric="accuracy", title="Breast cancer classifier")

**Why does the validation loss sit *below* the training loss at first?** This confuses
almost everyone the first time, and it is not a bug. Dropout is active during training but
switched off at validation time, so the training loss is measured on a deliberately
handicapped network while the validation loss is measured on the full one. The two curves
converge as training proceeds.

### 2.6 Evaluate on the test set

This is the first and only time the test set is used. Everything up to now — the
architecture, the dropout rate, the number of epochs — was decided on the validation set.

In [ ]:
test_loss, test_acc, test_auc = clf.evaluate(X_test, y_test, verbose=0)
print(f"Test loss      : {test_loss:.4f}")
print(f"Test accuracy  : {test_acc:.4f}")
print(f"Test AUC       : {test_auc:.4f}\n")

y_prob = clf.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ax.imshow(cm, cmap="Purples")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                fontsize=15, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], cancer.target_names)
ax.set_yticks([0, 1], cancer.target_names)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()

Worth pausing on: the network outputs a **probability**, and `0.5` is just one possible
threshold. In a medical setting a false negative (calling a malignant mass benign) costs
far more than a false positive, so you would lower the threshold deliberately and accept
more false alarms. The model does not make that trade-off for you.

## 3 · Regression — California housing

20,640 California districts, 8 numeric features (median income, house age, average rooms,
population, latitude, longitude, …). The target is the **median house value** of the
district, in hundreds of thousands of dollars.

The target is now a continuous number, so the output distribution is **Gaussian** — which,
from Part 04, means a **single linear output unit** (no activation) and **mean squared
error** as the loss. Maximising the log-likelihood of a Gaussian *is* minimising MSE.

### 3.1 Load, split, scale

In [ ]:
housing = fetch_california_housing()
Xh, yh = housing.data, housing.target

print("Feature matrix:", Xh.shape)
print("Features:", list(housing.feature_names))
print(f"Target range: {yh.min():.2f} to {yh.max():.2f}  (in $100,000s)")

Xh_tmp, Xh_test, yh_tmp, yh_test = train_test_split(
    Xh, yh, test_size=0.20, random_state=SEED)
Xh_train, Xh_val, yh_train, yh_val = train_test_split(
    Xh_tmp, yh_tmp, test_size=0.25, random_state=SEED)

h_scaler = StandardScaler().fit(Xh_train)
Xh_train = h_scaler.transform(Xh_train)
Xh_val   = h_scaler.transform(Xh_val)
Xh_test  = h_scaler.transform(Xh_test)
print(f"\ntrain {Xh_train.shape[0]}   val {Xh_val.shape[0]}   test {Xh_test.shape[0]}")

### 3.2 Build and compile

Almost the same network — and that is the point. Only two things change, both forced by
the task rather than chosen by taste:

- the **output layer** has no activation, so it can produce any real number;
- the **loss** is `mse` instead of `binary_crossentropy`.

We also use a gentler dropout rate (0.2). This dataset is much larger than the cancer one,
so there is less to overfit and less regularization is needed.

In [ ]:
reg = keras.Sequential([
    keras.Input(shape=(Xh_train.shape[1],)),
    layers.Dense(64, activation="relu", kernel_initializer="he_normal"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu", kernel_initializer="he_normal"),
    layers.Dropout(0.2),
    layers.Dense(1),                      # linear output — Gaussian p(y|x)
], name="california_housing_regressor")

reg.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"],
)
reg.summary()

### 3.3 Train

In [ ]:
early_stop_reg = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)

history_reg = reg.fit(
    Xh_train, yh_train,
    validation_data=(Xh_val, yh_val),
    epochs=200,
    batch_size=64,
    callbacks=[early_stop_reg],
    verbose=0,
)
print(f"Stopped after {len(history_reg.history['loss'])} epochs.")

In [ ]:
plot_history(history_reg, metric="mae", title="California housing regressor")

### 3.4 Evaluate

In [ ]:
yh_pred = reg.predict(Xh_test, verbose=0).ravel()

mse  = mean_squared_error(yh_test, yh_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(yh_test, yh_pred)
r2   = r2_score(yh_test, yh_pred)

print(f"Test MSE  : {mse:.4f}")
print(f"Test RMSE : {rmse:.4f}   (~ ${rmse * 100_000:,.0f})")
print(f"Test MAE  : {mae:.4f}   (~ ${mae * 100_000:,.0f})")
print(f"Test R²   : {r2:.4f}")

RMSE and MAE differ for a reason you met in Part 04: squared error is pulled around by
large outliers, absolute error is not. The gap between the two numbers is a rough measure
of how many districts the model gets badly wrong.

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.scatter(yh_test, yh_pred, s=6, alpha=0.25, color="#2D1B4E")
lims = [yh_test.min(), yh_test.max()]
ax.plot(lims, lims, color="#E8833A", lw=2, label="perfect prediction")
ax.set_xlabel("actual value  ($100,000s)")
ax.set_ylabel("predicted value  ($100,000s)")
ax.set_title("Predicted vs actual")
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

The horizontal band along the top is worth noticing: the target was **capped** at 5.0
(i.e. $500,000) when the dataset was built, so every genuinely expensive district is
recorded at exactly that ceiling. The model cannot predict past a limit that the data
itself does not contain — a good reminder that inspecting your target distribution is part
of the job, not an optional extra.

## 4 · Where each choice came from

| Choice in the code | Session part | Why |
|---|---|---|
| `Dense` layers chained together | Part 03 | Depth = length of the chain, width = units per layer |
| `activation="relu"` | Part 04 | The default recommendation for hidden units |
| `activation="sigmoid"` + `binary_crossentropy` | Part 04 | Bernoulli output → negative log-likelihood |
| no output activation + `mse` | Part 04 | Gaussian output → maximising log-likelihood = minimising MSE |
| `kernel_initializer="he_normal"` | Part 06 | N(0, 2/m), the scheme matched to ReLU |
| `Dropout(...)` | Part 05 | Approximates a bagged ensemble of thinned sub-networks |
| `EarlyStopping(restore_best_weights=True)` | Part 05 | Return the parameters with the lowest validation error |
| `StandardScaler` | Part 06 | Un-scaled features produce an ill-conditioned loss surface |
| `Adam(learning_rate=1e-3)` | Part 06 | Momentum + RMSProp + bias correction, with the suggested default |
| `batch_size=32` / `64` | Part 06 | Mini-batch SGD — the practical compromise |

## 5 · Things to try

Each of these takes one edit and one re-run. Watch the **gap** between the training and
validation curves — that gap is the thing every technique in Part 05 exists to close.

1. **Remove the dropout layers.** Does the training loss fall further? Does the validation
   loss follow it down, or start climbing?
2. **Set `patience=200`** so early stopping effectively never fires. Where does the
   validation curve turn, and how much worse does the test score get?
3. **Break the scaling.** Comment out the three `transform` lines in §2.2 and retrain.
   This is the single most dramatic change in the notebook.
4. **Swap the optimizer** for `keras.optimizers.SGD(learning_rate=0.01)`, then
   `SGD(learning_rate=0.01, momentum=0.9)`, then Adam. Compare epochs-to-converge.
5. **Use `kernel_initializer="zeros"`** in the classifier. Symmetry never breaks and the
   network cannot learn — the failure mode from Part 06, in about twenty seconds.
6. **Go deeper**: add two more hidden layers. Does it help, or does it just get harder to
   optimize?

---

🎉 **That's both models.** The theory from the session, start to finish, in about a hundred lines of Keras.